In [ ]:
# LOADING IN LIBRARIES
import json
from pathlib import Path
from pprint import pprint
import os
import getpass
from langchain_community.vectorstores import Chroma
import chromadb
import runhouse as rh
from sentence_transformers import SentenceTransformer
from langchain_core.documents import Document

In [ ]:
# LOADING IN DATA
file_path = './winemag-data-130k-v2.json'
data = json.loads(Path(file_path).read_text())

In [ ]:
print(data[0])
print(len(data))

In [ ]:
# CREATING DOCUMENTS SO DATA CAN BE EMBEDDED
documents = []
metadatas = []

i = 0
for entry in data: 
    # Construct each document
    documents.append(entry['description']) # Grabbing only the description as a string to be used for embedding
    metadatas.append(entry)
    i += 1
    if i == 5000:
        break

# Verify the number of documents created
print(f"Number of documents created: {len(documents)}")

In [ ]:
print(documents[:5])

In [ ]:
# EMBEDDING
# same model that Chromadb uses - intializing embedding model
model = SentenceTransformer('all-MiniLM-L6-v2')

#converts text into numerical embeddings - converts each document into a vector representation (embedding)
document_embeddings = model.encode(documents)

# each document needs a unique id to be used with chromadb
ids = [f"doc_{i}" for i in range(len(document_embeddings))]
# NOTE SINCE THERE ARE SO MANY DOCUMENTS CONSIDER SAVING EMBEDDINGS SO I DONT EVER HAVE TO RUN THIS AGAIN
# ALSO NOTE THAT YOU CAN RUN MULTIPLE DOCUMENTS AT THE SAME TIME WITH CERTAIN PROGRAMS 
# (ASYNCHRONOUS PROGRAMMING)

In [ ]:
print(len(document_embeddings))
print(metadatas[0])
print(documents[0])
print(len(ids))

In [ ]:
# Issue with metadata is that it is blank in some of the categories registered as the value None
#  this doesn't work with Chromadb so NA is used to replace the blanks so it will work in the vectorDB
print(type(metadatas))
print(metadatas[0])
if not isinstance(metadatas[0], (str, int, float, bool)):
    print(f"Invalid metadata value: {metadatas} of type {type(metadatas).__name__}/n")

# Replace None with default value for each dictionary in the list - this is needed for Chromadb
metadataFilled = [
    {key: (value if value is not None else 'NA') for key, value in metadata_item.items()}
    for metadata_item in metadatas
]

# Output the result
print(metadataFilled[0])

In [ ]:
# NEED TO CLEAR OLD COLLECTION IF I WANT TO WRITE OVER IT WITH NEW COLLECTION
chroma_client.delete_collection(name="wine_data_collection")

In [ ]:
# # VECTORIZING DATA

chroma_client = chromadb.Client() # inializes chromadb so we can connect our collection to the vector storage

collection = chroma_client.create_collection(name="wine_data_collection") #creating collection - this is essentially a vector db that can store embeddings, queries, and documents
# NOTE - The collection name must start and end with a lowercase letter

collection.add(
    ids=ids,  #list of unqiue ids connected to documents
    embeddings=document_embeddings,  # List of embeddings
    metadatas=metadataFilled,  #should be a list of dictionaries
    documents=documents  ##documentString #optional
    # consider changing the distance method of the embedding space to see how this impacts the model
)

# retriever = db.as_retriever()

# If Chroma is passed a list of documents, it will automatically tokenize and embed them with the
#  collection's embedding function (the default will be used if none was supplied at collection creation)
# Chroma will also store the documents themselves.

In [ ]:
# Lets me peek at collection to ensure that everything is loading in properly
print(collection.peek())

In [ ]:
# Query returns the closest match to each query embedding (in order)
results = collection.query(
    query_texts=["Can you recommend me a Portuguese wine?"], #"can you give me a smoky wine?"
    n_results=1
    # where={"variety": "Cabernet Sauvignon"}, # <- maybe use this for certain wines or regions
    # where_document={"$contains":"tobacco"}  #<- this would make it a traditional search
    #include=["documents"]
) #note region 1 will contain Napa Valley, region 2 will contain napa
results